In [63]:
import subprocess
import platform
import getpass

## Получение серийного номера материнской платы

In [64]:
if platform.system() == "Darwin":
    cmd = "ioreg -l | grep IOPlatformSerialNumber"
    raw_bytes = subprocess.check_output(cmd, shell=True)
    serial = raw_bytes.decode("utf-8").split("=")[-1].strip().strip('"')
else:
    cmd = "wmic baseboard get serialnumber"
    raw_bytes = subprocess.check_output(cmd, shell=True)
    serial = raw_bytes.decode("utf-8").split("\n")[1].strip()

print(serial)

HF0XF6K0WG


## Получение архитектуры процессора, название ядра ОС, версию операционной системы

In [65]:
arch = platform.machine()
os_name = platform.system()
os_ver = platform.release()

print(arch, os_name, os_ver)

arm64 Darwin 25.6.0


## Получение имени компьютера (хоста) и имени текущего пользователя

In [66]:
host = platform.node()
user = getpass.getuser()

print(host, user)

MacBook-Air-M misak2006mail.ru


## Склейка всех 6 параметров в одну строку

In [67]:
system_info = f"{serial}|{arch}|{os_name}|{os_ver}|{host}|{user}"
print(system_info)

HF0XF6K0WG|arm64|Darwin|25.6.0|MacBook-Air-M|misak2006mail.ru


## Мой ХЭШ

In [68]:
def crc32(data: bytes) -> int:
    crc = 0xFFFFFFFF
    for byte in data:
        crc ^= byte
        for _ in range(8):
            if crc & 1:
                crc = (crc >> 1) ^ 0xEDB88320
            else:
                crc >>= 1
    return crc ^ 0xFFFFFFFF


def get_hash(text: str) -> str:
    h = crc32(text.encode('utf-8'))
    return f'{h:08X}'

In [69]:
print(get_hash(system_info))

8E2E233B


## Создаем файл с хэшем

In [70]:
with open("license.key", "w") as f:
    f.write(get_hash(system_info))

print("Лицензия установлена")

Лицензия установлена


## Проверяем, что это создалось

In [71]:
print(open("license.key").read())

8E2E233B


In [72]:
def check_license():
    with open("license.key") as f:
        saved = f.read().strip()
    current = get_hash(system_info)
    if saved == current:
        print("Лицензия действительна. Программа работает.")
        return True
    else:
        print("НЕЛЕГАЛЬНОЕ ИСПОЛЬЗОВАНИЕ! Программа заблокирована.")
        return False

check_license()

Лицензия действительна. Программа работает.


True

## Тестируем типо это другой компьютер

In [73]:
# Имитация чужого компьютера: меняем хост
system_info = f"{serial}|{arch}|{os_name}|{os_ver}|Other-PC|{user}"
print("Новая строка:", system_info)

check_license()

Новая строка: HF0XF6K0WG|arm64|Darwin|25.6.0|Other-PC|misak2006mail.ru
НЕЛЕГАЛЬНОЕ ИСПОЛЬЗОВАНИЕ! Программа заблокирована.


False